In [ ]:
from pathlib import Path

base_path = Path("SemEval-2020/datasets")

articles_path = base_path / "train-articles"
si_labels_path = base_path / "train-labels-task1-span-identification"
tc_labels_path = base_path / "train-labels-task2-technique-classification"

In [ ]:
articles = {}

for txt in articles_path.glob("*.txt"):
    article_id = txt.stem.replace("article", "")
    articles[article_id] = txt.read_text(encoding="utf-8")

print(f"Статей {len(articles)}")

In [ ]:
si_labels = {}

for label in si_labels_path.glob("*.labels"):
    article_id = label.stem.replace("article", "").replace(".task1-SI", "")
    spans = []

    for line in label.read_text(encoding="utf-8").splitlines():
        _, start, end = line.split("\t")
        spans.append((int(start), int(end)))

    si_labels[article_id] = spans

print(next(iter(si_labels.items())))

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

# MODEL_NAME = "bert-base-uncased"
MODEL_NAME = "FacebookAI/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class BertSpanDataset(Dataset):
    def __init__(self, articles, si_labels, tokenizer, max_len=512):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len
        for article_id, text in articles.items():
            spans = si_labels.get(article_id, [])
            encoding = tokenizer(
                text,
                return_offsets_mapping=True,
                truncation=True,
                max_length=max_len,
                padding="max_length"
            )
            offsets = encoding["offset_mapping"]
            labels = []
            for start, end in offsets:
                if start == end:
                    labels.append(-100)
                    continue
                label = 0  # O
                for span_start, span_end in spans:
                    if start >= span_start and start < span_end:
                        if start == span_start:
                            label = 1  # B
                        else:
                            label = 2  # I
                        break
                labels.append(label)
            encoding["labels"] = labels
            self.samples.append({k: torch.tensor(v) for k, v in encoding.items()})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(list(articles.keys()), test_size=0.2, random_state=42)

train_articles = {k: articles[k] for k in train_ids}
test_articles = {k: articles[k] for k in test_ids}

train_labels = {k: si_labels.get(k, []) for k in train_ids}
test_labels = {k: si_labels.get(k, []) for k in test_ids}

train_dataset = BertSpanDataset(train_articles, train_labels, tokenizer)
test_dataset = BertSpanDataset(test_articles, test_labels, tokenizer)

In [ ]:
id2label = {
    0: "O",
    1: "B",
    2: "I"}

def print_bert_example(dataset, tokenizer, idx=0, max_tokens=30):
    sample = dataset[idx]
    input_ids = sample["input_ids"]
    labels = sample["labels"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    printed = 0
    for token, label in zip(tokens, labels):
        label = label.item()
        if label == -100:
            continue  
        print({
            "token": token,
            "label": id2label[label]})
        printed += 1
        if printed >= max_tokens:
            break
print_bert_example(train_dataset, tokenizer, idx=7)

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3)
model = model.to("cuda")

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

In [ ]:
import numpy as np

id2label = {0: "O", 1: "B", 2: "I"}
def bio_to_spans_from_logits(pred_ids):
    spans = []
    start = None
    for i, lab in enumerate(pred_ids):
        if lab == 1:
            if start is not None:
                spans.append((start, i))
            start = i
        elif lab == 2:
            if start is None:
                start = i
        else:
            if start is not None:
                spans.append((start, i))
                start = None
    if start is not None:
        spans.append((start, len(pred_ids)))

    return spans

In [ ]:
def merge_spans(spans):
  if not spans:
    return []
  spans = sorted(spans, key=lambda x: x[0])
  merged = [spans[0]]
  for current in spans[1:]:
      prev_start, prev_end = merged[-1]
      curr_start, curr_end = current
      if curr_start <= prev_end:
          merged[-1] = (prev_start, max(prev_end, curr_end))
      else:
          merged.append(current)
  return merged

In [ ]:
def compute_dataset_scores(all_pred, all_gold):
    #Реализация по формуле из SemEval 2020 Task 11
    S = sum(len(merge_spans(p)) for p in all_pred)
    T = sum(len(merge_spans(g)) for g in all_gold)
    if S == 0:
        precision = 0
    else:
        precision_sum = 0
        for pred_spans, gold_spans in zip(all_pred, all_gold):
            pred_spans = merge_spans(pred_spans)
            gold_spans = merge_spans(gold_spans)

            for s in pred_spans:
                for t in gold_spans:
                    overlap = max(0, min(s[1], t[1]) - max(s[0], t[0]))
                    if t[1] > t[0]:
                        precision_sum += overlap / (t[1] - t[0])
        precision = precision_sum / S
    if T == 0:
        recall = 0
    else:
        recall_sum = 0
        for pred_spans, gold_spans in zip(all_pred, all_gold):
            pred_spans = merge_spans(pred_spans)
            gold_spans = merge_spans(gold_spans)

            for s in pred_spans:
                for t in gold_spans:
                    overlap = max(0, min(s[1], t[1]) - max(s[0], t[0]))
                    if s[1] > s[0]:
                        recall_sum += overlap / (s[1] - s[0])
        recall = recall_sum / T
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    all_pred_spans = []
    all_gold_spans = []

    for pred, gold in zip(predictions, labels):
        mask = gold != -100
        pred = pred[mask]
        gold = gold[mask]

        pred_spans = bio_to_spans_from_logits(pred)
        gold_spans = bio_to_spans_from_logits(gold)

        all_pred_spans.append(pred_spans)
        all_gold_spans.append(gold_spans)

    precision, recall, f1 = compute_dataset_scores(all_pred_spans, all_gold_spans)

    # print(f"\nP: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()